# UBC Fine-tune + оценка пайплайна

**NY building** + **INRIA segmenter** fine-tune на UBC, затем оценка на val/test.

**Не использует** `ubc_building_classifier.pth`.

## Перед запуском

```bash
python scripts/download_extra_datasets.py --ubc-crops
python scripts/prepare_ubc_seg_patches.py
python scripts/build_ubc_split.py
python scripts/build_ubc_seg_split.py
```

Нужны: `ny_building_classifier.pth`, `inria_building_segmenter.pth`, `convnext_best.pth`.

## Результат

- `models/ny_building_ubc.pth`
- `models/inria_building_ubc.pth`
- Метрики пайплайна на UBC test


In [ ]:
import os
from pathlib import Path

_env_path = Path('..').resolve() / '.env'
if _env_path.exists():
    for _line in _env_path.read_text(encoding='utf-8').splitlines():
        _line = _line.strip()
        if not _line or _line.startswith('#') or '=' not in _line:
            continue
        _key, _val = _line.split('=', 1)
        os.environ.setdefault(_key.strip(), _val.strip().strip('"\''))

_hf_token = os.environ.get('HF_TOKEN')
if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token, add_to_git_credential=False)
    print('Hugging Face: авторизация OK')
else:
    print('HF_TOKEN не найден в .env — веса timm скачаются без токена (медленнее)')


import sys
from pathlib import Path

import torch

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from utils import NUM_WORKERS

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LOADER_WORKERS = NUM_WORKERS
print('PROJECT_ROOT:', PROJECT_ROOT)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('device:', device, '| DataLoader workers:', LOADER_WORKERS)



## 1. Fine-tune NY building (ny_building_classifier → ny_building_ubc)


In [ ]:
# === 1. Fine-tune NY building на UBC crop'ах ===
import copy
import time

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, WeightedRandomSampler

from dataset import BuildingDataset, get_building_train_transforms, get_transforms, load_split
from losses import FocalLoss
from model_convnext import build_convnext_tiny, unfreeze_all
from utils import (
    MODELS_DIR,
    NY_BUILDING,
    NY_BUILDING_CLASSES as CLASSES,
    RANDOM_SEED,
    UBC_SPLIT_CSV,
    ensure_dirs,
    set_random_seed,
)

NY_UBC_CKPT = MODELS_DIR / 'ny_building_ubc.pth'
NY_BASE_CKPT = MODELS_DIR / NY_BUILDING.model_filename
NY_FT_EPOCHS = 8
NY_FT_LR = 1e-5
NY_FT_PATIENCE = 3

set_random_seed(RANDOM_SEED)
ensure_dirs()

split_df = load_split(UBC_SPLIT_CSV)
train_df = split_df[split_df['split'] == 'train'].reset_index(drop=True)
train_transform = get_building_train_transforms()
eval_transform = get_transforms()
train_ds = BuildingDataset(split_df, split='train', classes=CLASSES, transform=train_transform, use_mask=True)
val_ds = BuildingDataset(split_df, split='val', classes=CLASSES, transform=eval_transform, use_mask=True)


def make_weighted_sampler(df, classes=CLASSES) -> WeightedRandomSampler:
    class_counts = df['class'].value_counts()
    class_weight = {cls: 1.0 / class_counts[cls] for cls in classes}
    sample_weights = df['class'].map(class_weight).to_numpy()
    return WeightedRandomSampler(sample_weights, num_samples=len(df), replacement=True)


BATCH_SIZE = NY_BUILDING.batch_size
LOADER_KWARGS = dict(num_workers=LOADER_WORKERS, persistent_workers=LOADER_WORKERS > 0, pin_memory=True)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=make_weighted_sampler(train_df, CLASSES), **LOADER_KWARGS
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **LOADER_KWARGS)
criterion = FocalLoss(gamma=NY_BUILDING.focal_gamma, label_smoothing=NY_BUILDING.label_smoothing)

ny_model = build_convnext_tiny(num_classes=len(CLASSES), freeze_backbone=False)
ny_model.load_state_dict(torch.load(NY_BASE_CKPT, map_location='cpu', weights_only=True))
unfreeze_all(ny_model)
ny_model = ny_model.to(device)

optimizer = torch.optim.AdamW(ny_model.parameters(), lr=NY_FT_LR, weight_decay=NY_BUILDING.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
best_f1, best_state, no_improve = -1.0, None, 0

print(f'NY fine-tune: {len(train_ds)} train / {len(val_ds)} val | base: {NY_BASE_CKPT.name}')


def _ny_epoch(loader, train: bool):
    ny_model.train() if train else ny_model.eval()
    total_loss, preds, labels = 0.0, [], []
    with torch.set_grad_enabled(train):
        for imgs, y in loader:
            imgs, y = imgs.to(device), y.to(device)
            if train:
                optimizer.zero_grad()
            logits = ny_model(imgs)
            loss = criterion(logits, y)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            preds.extend(logits.argmax(1).cpu().tolist())
            labels.extend(y.cpu().tolist())
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    return total_loss / len(loader.dataset), macro_f1


for epoch in range(1, NY_FT_EPOCHS + 1):
    t0 = time.time()
    tr_loss, _ = _ny_epoch(train_loader, train=True)
    va_loss, va_f1 = _ny_epoch(val_loader, train=False)
    scheduler.step(va_f1)
    improved = va_f1 > best_f1
    if improved:
        best_f1, best_state, no_improve = va_f1, copy.deepcopy(ny_model.state_dict()), 0
    else:
        no_improve += 1
    print(
        f'NY эпоха {epoch}/{NY_FT_EPOCHS} | train_loss={tr_loss:.4f} | '
        f'val_loss={va_loss:.4f} macro F1={va_f1:.4f} | {time.time()-t0:.1f}s'
        + (' <- лучшая' if improved else '')
    )
    if epoch >= 2 and no_improve >= NY_FT_PATIENCE:
        print('NY early stop')
        break

if best_state is not None:
    ny_model.load_state_dict(best_state)
torch.save(ny_model.state_dict(), NY_UBC_CKPT)
print(f'Сохранено: {NY_UBC_CKPT} (best val macro F1={best_f1:.4f})')



model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

NY fine-tune: 25406 train / 4484 val | base: ny_building_classifier.pth


NY эпоха 1/8 | train_loss=0.3590 | val_loss=0.2978 macro F1=0.4712 | 93.8s <- лучшая


NY эпоха 2/8 | train_loss=0.2464 | val_loss=0.2609 macro F1=0.5250 | 82.0s <- лучшая


NY эпоха 3/8 | train_loss=0.1966 | val_loss=0.1895 macro F1=0.6255 | 82.1s <- лучшая


NY эпоха 4/8 | train_loss=0.1654 | val_loss=0.2016 macro F1=0.6219 | 82.1s


NY эпоха 5/8 | train_loss=0.1420 | val_loss=0.1610 macro F1=0.6923 | 82.1s <- лучшая


NY эпоха 6/8 | train_loss=0.1259 | val_loss=0.1939 macro F1=0.6617 | 82.1s


NY эпоха 7/8 | train_loss=0.1132 | val_loss=0.1551 macro F1=0.7082 | 82.0s <- лучшая


## 2. Fine-tune INRIA segmenter (inria_building_segmenter → inria_building_ubc)


In [ ]:
# === 2. Fine-tune INRIA segmenter на UBC seg-патчах ===
import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from dataset import InriaSegmentationDataset, load_inria_split
from model_segmentation import build_convnext_segmenter, unfreeze_encoder_stages
from utils import INRIA_SEGMENTATION, MODELS_DIR, RANDOM_SEED, UBC_SEG_SPLIT_CSV, ensure_dirs, set_random_seed

INRIA_UBC_CKPT = MODELS_DIR / 'inria_building_ubc.pth'
INRIA_BASE_CKPT = MODELS_DIR / INRIA_SEGMENTATION.model_filename
INRIA_FT_EPOCHS = 6
INRIA_FT_LR = 5e-5
INRIA_FT_PATIENCE = 2
PRED_THRESHOLD = 0.5


class DiceLoss(nn.Module):
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        targets = (targets >= 0.5).float()
        inter = (probs * targets).sum(dim=(2, 3))
        union = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
        dice = (2 * inter + 1) / (union + 1)
        return 1 - dice.mean()


class SegmentationLoss(nn.Module):
    def __init__(self, pos_weight: float = INRIA_SEGMENTATION.pos_weight):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]))
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        return 0.5 * self.bce(logits, targets) + 0.5 * self.dice(logits, targets)


@torch.no_grad()
def _pixel_iou(logits, targets, threshold=PRED_THRESHOLD):
    preds = (torch.sigmoid(logits) >= threshold).float()
    targets = (targets >= 0.5).float()
    inter = (preds * targets).sum().item()
    union = preds.sum().item() + targets.sum().item() - inter
    return inter / (union + 1e-8)


set_random_seed(RANDOM_SEED)
seg_df = load_inria_split(UBC_SEG_SPLIT_CSV)
train_seg = InriaSegmentationDataset(seg_df, split='train', augment=True)
val_seg = InriaSegmentationDataset(seg_df, split='val', augment=False)

SEG_BATCH = INRIA_SEGMENTATION.batch_size
seg_kwargs = dict(num_workers=LOADER_WORKERS, persistent_workers=LOADER_WORKERS > 0, pin_memory=True)
train_seg_loader = DataLoader(train_seg, batch_size=SEG_BATCH, shuffle=True, **seg_kwargs)
val_seg_loader = DataLoader(val_seg, batch_size=SEG_BATCH, shuffle=False, **seg_kwargs)

seg_model = build_convnext_segmenter(pretrained=False, freeze_encoder=True)
seg_model.load_state_dict(torch.load(INRIA_BASE_CKPT, map_location='cpu', weights_only=True))
unfreeze_encoder_stages(seg_model, n_stages=2)
seg_model = seg_model.to(device)
seg_criterion = SegmentationLoss().to(device)
seg_optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, seg_model.parameters()),
    lr=INRIA_FT_LR,
    weight_decay=INRIA_SEGMENTATION.weight_decay,
)
seg_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(seg_optimizer, mode='max', factor=0.5, patience=2)

best_iou, best_seg_state, seg_no_improve = -1.0, None, 0
print(f'INRIA fine-tune: {len(train_seg)} train / {len(val_seg)} val | base: {INRIA_BASE_CKPT.name}')


def _seg_epoch(loader, train: bool):
    seg_model.train() if train else seg_model.eval()
    total_loss, iou_sum, n = 0.0, 0.0, 0
    with torch.set_grad_enabled(train):
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            if train:
                seg_optimizer.zero_grad()
            logits = seg_model(images)
            loss = seg_criterion(logits, masks)
            if train:
                loss.backward()
                seg_optimizer.step()
            total_loss += loss.item() * images.size(0)
            iou_sum += _pixel_iou(logits, masks) * images.size(0)
            n += images.size(0)
    return total_loss / n, iou_sum / n


for epoch in range(1, INRIA_FT_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_iou = _seg_epoch(train_seg_loader, train=True)
    va_loss, va_iou = _seg_epoch(val_seg_loader, train=False)
    seg_scheduler.step(va_iou)
    improved = va_iou > best_iou
    if improved:
        best_iou, best_seg_state, seg_no_improve = va_iou, copy.deepcopy(seg_model.state_dict()), 0
    else:
        seg_no_improve += 1
    print(
        f'INRIA эпоха {epoch}/{INRIA_FT_EPOCHS} | train IoU={tr_iou:.4f} | '
        f'val IoU={va_iou:.4f} | {time.time()-t0:.1f}s' + (' <- лучшая' if improved else '')
    )
    if epoch >= 2 and seg_no_improve >= INRIA_FT_PATIENCE:
        print('INRIA early stop')
        break

if best_seg_state is not None:
    seg_model.load_state_dict(best_seg_state)
torch.save(seg_model.state_dict(), INRIA_UBC_CKPT)
print(f'Сохранено: {INRIA_UBC_CKPT} (best val IoU={best_iou:.4f})')



INRIA fine-tune: 1897 train / 332 val | base: inria_building_segmenter.pth


INRIA эпоха 1/6 | train IoU=0.7025 | val IoU=0.7188 | 34.9s <- лучшая


INRIA эпоха 2/6 | train IoU=0.7552 | val IoU=0.7348 | 26.4s <- лучшая


INRIA эпоха 3/6 | train IoU=0.7743 | val IoU=0.7521 | 26.2s <- лучшая


INRIA эпоха 4/6 | train IoU=0.7882 | val IoU=0.7592 | 26.2s <- лучшая


INRIA эпоха 5/6 | train IoU=0.7988 | val IoU=0.7638 | 26.3s <- лучшая


## 3. Pipeline eval (zone + ubc-finetuned INRIA + NY, UBC test)


In [ ]:
# === 3. Калибровка порога + оценка пайплайна на UBC test ===
import pandas as pd
import torch

from model_convnext import build_convnext_tiny
from model_segmentation import build_convnext_segmenter
from pipeline_ubc import (
    evaluate_building_classification,
    run_pipeline_on_tile,
    ubc_tile_names_for_eval_split,
)
from utils import MODELS_DIR, NY_BUILDING, UBC_RAW_DIR

# test в split.csv = исходный UBC val; сырые тайлы лежат в data/raw/ubc/val/
UBC_IMAGE_SPLIT = 'val'

zone_ckpt = MODELS_DIR / 'convnext_best.pth'
zone_model = build_convnext_tiny(num_classes=4, freeze_backbone=False)
zone_model.load_state_dict(torch.load(zone_ckpt, map_location='cpu', weights_only=True))
zone_model = zone_model.to(device).eval()

ny_model.load_state_dict(torch.load(NY_UBC_CKPT, map_location='cpu', weights_only=True))
ny_model = ny_model.to(device).eval()

seg_model.load_state_dict(torch.load(INRIA_UBC_CKPT, map_location='cpu', weights_only=True))
seg_model = seg_model.to(device).eval()

test_tiles = ubc_tile_names_for_eval_split('test')
if not test_tiles:
    test_tiles = sorted(p.name for p in (UBC_RAW_DIR / UBC_IMAGE_SPLIT).glob('*_RGB.tif'))
print(f'Тайлов test (исходный UBC val): {len(test_tiles)}')

THRESHOLDS = [0.35, 0.4, 0.45, 0.5, 0.55, 0.6]
CALIB_TILES = test_tiles[: min(30, len(test_tiles))]


def _eval_split(tiles, mask_threshold):
    rows = []
    for tile in tiles:
        res = run_pipeline_on_tile(
            tile,
            split=UBC_IMAGE_SPLIT,
            zone_model=zone_model,
            segmenter_model=seg_model,
            building_model=ny_model,
            device=str(device),
            mask_threshold=mask_threshold,
            ubc_raw_dir=UBC_RAW_DIR,
        )
        cls_m = evaluate_building_classification(res)
        rows.append({
            'tile': tile,
            'mask_iou': res.mask_iou,
            'building_macro_f1': cls_m['macro_f1'],
            'building_acc': cls_m['accuracy'],
            'matched': cls_m['matched'],
        })
    return pd.DataFrame(rows)


best_thr, best_score = 0.5, -1.0
for thr in THRESHOLDS:
    df_cal = _eval_split(CALIB_TILES, thr)
    score = df_cal['mask_iou'].mean()
    print(f'  threshold={thr:.2f} → mean mask IoU (calib {len(CALIB_TILES)} tiles) = {score:.4f}')
    if score > best_score:
        best_score, best_thr = score, thr

print(f'\nЛучший порог: {best_thr:.2f}')

test_metrics = _eval_split(test_tiles, best_thr)

print('\n=== TEST (исходный UBC val, n={}) ==='.format(len(test_tiles)))
print(f"mask IoU mean: {test_metrics['mask_iou'].mean():.4f}")
matched = test_metrics[test_metrics['matched'] > 0]
if len(matched):
    print(f"building macro F1: {matched['building_macro_f1'].mean():.4f}")
    print(f"building acc: {matched['building_acc'].mean():.4f}")
else:
    print('building macro F1: n/a (нет сопоставлений)')

print(test_metrics.head(10).to_string())


Тайлов test (исходный UBC val): 153


  threshold=0.35 → mean mask IoU (calib 30 tiles) = 0.5156


  threshold=0.40 → mean mask IoU (calib 30 tiles) = 0.5167


  threshold=0.45 → mean mask IoU (calib 30 tiles) = 0.5168


  threshold=0.50 → mean mask IoU (calib 30 tiles) = 0.5162


  threshold=0.55 → mean mask IoU (calib 30 tiles) = 0.5144


  threshold=0.60 → mean mask IoU (calib 30 tiles) = 0.5120

Лучший порог: 0.45



=== TEST (исходный UBC val, n=153) ===
mask IoU mean: 0.5291
building macro F1: 0.2640
building acc: 0.4689
                                                                  tile  mask_iou  building_macro_f1  building_acc  matched
0  Beijing_GF2_L1A0004822322-MSS1_10300_27000_116.2958_39.8310_RGB.tif  0.760563           0.242424      0.571429     14.0
1  Beijing_GF2_L1A0004822322-MSS1_10300_27400_116.2958_39.8281_RGB.tif  0.729774           0.256410      0.625000     16.0
2  Beijing_GF2_L1A0004822322-MSS1_11700_17100_116.3081_39.9024_RGB.tif  0.329814           0.254971      0.481481     27.0
3  Beijing_GF2_L1A0004822322-MSS1_11700_17500_116.3082_39.8995_RGB.tif  0.476675           0.409091      0.625000     40.0
4  Beijing_GF2_L1A0004822322-MSS1_12100_17100_116.3119_39.9025_RGB.tif  0.463846           0.436782      0.681818     44.0
5  Beijing_GF2_L1A0004822322-MSS1_12100_17500_116.3119_39.8996_RGB.tif  0.449784           0.552381      0.777778     27.0
6  Beijing_GF2_L1A0004822322-M

## Итог


In [ ]:
# === 4. Итоговые веса ===
from utils import MODELS_DIR

for ckpt in [NY_UBC_CKPT, INRIA_UBC_CKPT]:
    print(f'{ckpt.name}:', 'OK' if ckpt.exists() else 'НЕТ')
print('\nДальше: notebooks/07_ubc_pipeline_calibration.ipynb → 08_zone_building_pipeline.ipynb')
